# XGBoost 再学習ノートブック v3

| セル  | 内容 |
|-------|------|
| セル1 | Google Drive マウント |
| セル2 | src/ 強制アップデート（GitHub最新） |
| セル3 | speed_index キャッシュ再構築 → 学習データ生成 |
| セル4 | 複勝XGB再学習 → キャリブレーション |
| セル4b | ランキングモデル学習（rank:pairwise / rank:ndcg） |
| セル4c | Temperature Scaling → モデル比較 |
| セル5 | 統合テスト（cal_prob合計・重要度確認） |
| セル6 | 再学習モデルを GitHub main にプッシュ（本番反映） |

> 使い方: Colabの「ファイル → ノートブックを開く → GitHub」から本ファイルを開き、セル1〜6を順に実行する。
> ⚠ セル6を実行しないと、Driveで再学習したモデルは本番（GitHub Actions予想）に反映されない。
> 前提: Colabシークレット(🔑)に GITHUB_PAT を登録し、本ノートの「ノートブックからのアクセス」をONにすること。

In [ ]:
# ── セル1: セットアップ ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'✅ BASE_DIR: {BASE_DIR}')

In [ ]:
# ── セル2: src/ 強制アップデート（GitHub最新コードを取得）────────
import urllib.request, time as _time
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())
files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py',
    'src/tools/calibrate.py', 'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py', 'src/tools/build_training_data.py',
    'src/tools/train_xgb.py', 'src/tools/calibrate_xgb.py',
    'src/features/engine.py', 'src/features/speed_index.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py',
    'src/models/calibration_xgb.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py',
    'src/betting/ev_filter.py', 'src/betting/app_json.py',
    'src/tools/train_ranking_model.py',
    'src/tools/compare_models.py',
    'src/betting/race_simulator.py',
    'src/betting/ev_calculator.py',
    'src/betting/rating_calibration.py',
    'src/betting/payout_estimator.py',
    'src/betting/dual_model.py',
    'src/betting/bet_optimizer.py',
    'src/betting/shadow.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
    print(f'OK {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# 主要特徴量の存在確認
with open(f'{BASE_DIR}/src/features/engine.py') as _f:
    _eng_src = _f.read()
for _kw in ['f_member_level_avg', 'f_speed_fig_last', 'f_speed_fig_avg', 'f_finish_time_avg', '±200']:
    _ok = _kw in _eng_src
    print(f'  {"✅" if _ok else "❌"} engine.py に {_kw} {"あり" if _ok else "なし! ← 要確認"}')
print('done')

In [ ]:
# ── セル3: speed_index キャッシュ再構築 → 学習データ生成 ────────
# speed_index_cache.pkl を最新 history.db から再構築してから CSV 生成
from src.features.speed_index import rebuild_speed_index_cache
rebuild_speed_index_cache(BASE_DIR)

for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.build_training_data import build_training_data
df_all = build_training_data(BASE_DIR)

# 特徴量充足率確認
print('\n── 特徴量充足率確認 ──')
CHECK_COLS = [
    'f_member_level_avg', 'f_finish_time_avg', 'f_time_diff_avg',
    'f_speed_fig_last', 'f_speed_fig_avg', 'f_speed_fig_max',
]
for c in CHECK_COLS:
    if c in df_all.columns:
        pct = 100 * df_all[c].notna().sum() / len(df_all)
        print(f'  ✅ {c}: {pct:.1f}%')
    else:
        print(f'  ❌ {c}: 列なし')
print(f'\n総列数: {len(df_all.columns)}  総行数: {len(df_all)}')

In [ ]:
# ── セル4: XGB再学習 → キャリブレーション ───────────────────────
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_xgb import train_xgb
from src.tools.calibrate_xgb import run_xgb_calibration

# ① 再学習
metrics = train_xgb(BASE_DIR,
    train_end='2026-05-31',
    val_start='2026-06-01',
    val_end='2026-06-15')
print(metrics)

# ② AUC 0.75以上ならキャリブレーション
if metrics.get('auc', 0) >= 0.75:
    run_xgb_calibration(BASE_DIR)
    print('✅ キャリブレーション完了')
else:
    print(f'⚠ AUC={metrics.get("auc", 0):.4f} < 0.75 → キャリブレーションスキップ')

In [ ]:
# ── セル4b: ランキングモデル学習（rank:pairwise / rank:ndcg）────────
# 複勝モデル(セル4)と同じ特徴量を使い、着順ランキングを学習する対抗モデル。
# 学習データ: horse_features.csv（セル3で生成済み）
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_ranking_model import train_ranking_model

print('── rank:pairwise 学習 ──')
train_ranking_model(BASE_DIR, objective='rank:pairwise', model_suffix='pairwise',
                    train_end='2026-05-31',
                    val_start='2026-06-01', val_end='2026-06-15')

print()
print('── rank:ndcg 学習 ──')
train_ranking_model(BASE_DIR, objective='rank:ndcg', model_suffix='ndcg',
                    train_end='2026-05-31',
                    val_start='2026-06-01', val_end='2026-06-15')


In [ ]:
# ── セル4c: Temperature Scaling + モデル比較 ────────────────────────
# 各モデルの出力スケールを統一（ECE最小のT値を探索）してから
# 複数Val期間で命中率・ROI・ECEを比較する。
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.betting.rating_calibration import calibrate_all_models
from src.tools.compare_models import compare_all_models, print_comparison

# Step 1: 温度校正（rating_temperature.json に保存）
print('── Temperature Scaling ──')
calibrate_all_models(BASE_DIR,
                     val_start='2026-04-01',
                     val_end='2026-05-31',
                     n_sims=5000)

# Step 2: 複数期間でモデル比較
print()
print('── モデル比較 ──')
metrics = compare_all_models(BASE_DIR, [
    ('2026-06-01', '2026-06-15'),   # 直近Val
    ('2026-03-01', '2026-03-31'),   # 春
    ('2025-09-01', '2025-09-30'),   # 秋
], n_sims=5000)

print_comparison(metrics)


In [ ]:
# ── セル5: 統合テスト ──────────────────────────────────────────
import pickle, json, numpy as np, pandas as pd

with open(f'{BASE_DIR}/data/xgb_fukusho_model.pkl', 'rb') as f: model = pickle.load(f)
with open(f'{BASE_DIR}/data/xgb_feature_cols.json')        as f: cols  = json.load(f)['feature_cols']
with open(f'{BASE_DIR}/data/xgb_calibrator.pkl', 'rb')     as f: calib = pickle.load(f)

print(f'✅ モデル特徴量数: {len(cols)}')
print(f'✅ キャリブレーター: {type(calib).__name__}')

# ダミー16頭で cal_prob 合計確認
dummy = pd.DataFrame([{c: np.random.normal(5.0, 1.5) for c in cols} for _ in range(16)])
raw_probs = model.predict_proba(dummy)[:, 1]
cal_probs = np.array(calib.transform(raw_probs))

print(f'\n── ダミー16頭テスト ──')
print(f'  raw_prob range: {raw_probs.min():.3f} ~ {raw_probs.max():.3f}')
print(f'  cal_prob 合計:  {cal_probs.sum():.3f}  (2.8〜3.2 が正常)')
print(f'  合計チェック    {"✅" if 2.8 <= cal_probs.sum() <= 3.2 else "⚠ 異常値"}')
print(f'  分散チェック    {"✅" if raw_probs.max() - raw_probs.min() > 0.3 else "⚠ 分散小さい"}')

# 特徴量重要度トップ20
_speed_fig = {'f_speed_fig_last', 'f_speed_fig_avg', 'f_speed_fig_max'}
_member    = {'f_member_level_avg', 'f_member_level_max', 'f_member_level_last'}
_time_feat = {'f_finish_time_avg', 'f_time_diff_avg'}
imps = sorted(zip(cols, model.feature_importances_), key=lambda x: x[1], reverse=True)[:20]
print('\n── 特徴量重要度 Top 20 ──')
for name, imp in imps:
    if name in _speed_fig:   mark = '★スピード指数'
    elif name in _member:    mark = '★メンバーLv'
    elif name in _time_feat: mark = '★タイム(±200m)'
    else:                    mark = ''
    print(f'  {name:<42} {imp*100:.2f}% {mark}')

In [ ]:
# ── セル6: 再学習モデルを GitHub main にプッシュ（Contents API・通常blob）──
# .gitattributes で LFS除外の3ファイルを GitHub に反映する。
# これを実行しないと、Driveで再学習したモデルが本番(GitHub Actions予想)に反映されない。
# 前提: Colabのシークレット(🔑)に GITHUB_PAT を登録し、このノートの「ノートブックからの
#       アクセス」をONにしておくこと（latest.json push と同じ PAT）。
import requests, base64, json as _json
from google.colab import userdata

GITHUB_PAT = userdata.get('GITHUB_PAT')
REPO  = 'hanagenuku/keiba_ai'
FILES = ['data/xgb_fukusho_model.pkl',
         'data/xgb_feature_cols.json',
         'data/xgb_calibrator.pkl',
         'data/xgb_ranking_pairwise.pkl',
         'data/xgb_ranking_ndcg.pkl',
         'data/xgb_ranking_feature_cols.json',
         'data/rating_temperature.json']

def push_file(relpath, pat, message):
    url = f'https://api.github.com/repos/{REPO}/contents/{relpath}'
    h = {'Authorization': f'token {pat}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(url, headers=h, params={'ref': 'main'})
    sha = r.json().get('sha') if r.status_code == 200 else None   # 更新にはSHA必須
    with open(f'{BASE_DIR}/{relpath}', 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    payload = {'message': message, 'content': content, 'branch': 'main'}
    if sha:
        payload['sha'] = sha
    r = requests.put(url, headers=h, json=payload)
    ok = r.status_code in (200, 201)
    print(('✅ ' if ok else f'⚠️ {r.status_code} ') + relpath +
          ('' if ok else f" → {r.json().get('message','')}"))
    return ok

n = len(_json.load(open(f'{BASE_DIR}/data/xgb_feature_cols.json'))['feature_cols'])
print(f'反映する特徴量数: {n}')
for rel in FILES:
    import os as _os
    if not _os.path.exists(f'{BASE_DIR}/{rel}'):
        print(f'⏭ スキップ（未生成）: {rel}')
        continue
    push_file(rel, GITHUB_PAT, f'model: retrain {n}feat')
print('完了 — 数分後にGitHubの data/xgb_feature_cols.json が更新される')